# Mapping migration


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

You will also need to import any libraries you are using in this notebook. In this notebook, you will need some additional packages to make interactive maps with vector data and access pre-defined month names.

In [2]:
### import packages

### libraries for dynamic mapping
import cartopy.crs as ccrs
import panel as pn
import hvplot.pandas
import holoviews as hv

### get month names
import calendar

import os 
import pathlib

## Step 5: Make an interactive map of GBIF observations

### Step 5A: Create a simplified `GeoDataFrame` for plotting

Plotting larger files can be time consuming. The code below will streamline plotting with `hvplot` by simplifying the geometry, projecting it to a Mercator projection that is compatible with `geoviews`, and cropping off areas in the Arctic:

1. Simplify the ecoregions with `.simplify(.05)`, and save it back to the `geometry` column.
2. Change the Coordinate Reference System (CRS) to Mercator with `.to_crs(ccrs.Mercator())`.
3. Use the plotting code that is already in the cell to check that the plotting runs quickly (less than a minute) and looks the way you want, making sure to change `gdf` to your `GeoDataFrame` name.

In [3]:
### simplify the geometry to speed up processing
ecoreg_gdf.geometry = ecoreg_gdf.simplify(
    0.05, preserve_topology= False)
### change the CRS to Mercator for mapping
ecoreg_gdf = ecoreg_gdf.to_crs(ccrs.Mercator())
### check that the plot runs in a reasonable amount of time
ecoreg_gdf.hvplot(geo = True, crs = ccrs.Mercator())

:Polygons   [Longitude,Latitude]

### Step 5B: Map migration over time

1. If applicable, replace any variable names with the names you defined previously.
2. Replace `column_name_used_for_ecoregion_color` and `column_name_used_for_slider` with the column names you wish to use.
3. Customize your plot with your choice of title, tile source, color map, and size.

Your plot will probably still change months very slowly in your Jupyter notebook, because it calculates each month’s plot as needed. Open up the saved HTML file to see faster performance!

In [4]:
### prep the data, then combine the occurrence data with the ecoregions

### rename columns in ecoregID to match occurrence_df
ecoreg_gdf = ecoreg_gdf.rename(columns = {'OBJECTID': 'ecoregion'})

### make ecoreg ID the index for merge with occurrence_df
ecoreg_gdf = ecoreg_gdf.set_index('ecoregion')

### join the occurrences with the plotting GeoDataFrame
occurrence_gdf = ecoreg_gdf.join(occurrence_df)

occurrence_gdf.head()

ECO_NAME  BIOME_NUM  \
ecoregion month                                                          
13.0      5      Alberta-British Columbia foothills forests        5.0   
          6      Alberta-British Columbia foothills forests        5.0   
          7      Alberta-British Columbia foothills forests        5.0   
17.0      4                     Allegheny Highlands forests        4.0   
          5                     Allegheny Highlands forests        4.0   

                                          BIOME_NAME     REALM ECO_BIOME_  \
ecoregion month                                                             
13.0      5                Temperate Conifer Forests  Nearctic       NE05   
          6                Temperate Conifer Forests  Nearctic       NE05   
          7                Temperate Conifer Forests  Nearctic       NE05   
17.0      4      Temperate Broadleaf & Mixed Forests  Nearctic       NE04   
          5      Temperate Broadleaf & Mixed Forests  Nearctic       NE04   

                 NNH  ECO_ID  SHAPE_LENG  SHAPE_AREA  \
ecoregion month                                        
13.0      5        2     345   62.333821   17.133639   
          6        2     345   62.333821   17.133639   
          7        2     345   62.333821   17.133639   
17.0      4        2     328   24.038587    7.958751   
          5        2     328   24.038587    7.958751   

                                          NNH_NAME    COLOR COLOR_BIO  \
ecoregion month                                                         
13.0      5      Nature Could Reach Half Protected  #5DAD4C   #458970   
          6      Nature Could Reach Half Protected  #5DAD4C   #458970   
          7      Nature Could Reach Half Protected  #5DAD4C   #458970   
17.0      4      Nature Could Reach Half Protected  #C8EBB1   #00734C   
          5      Nature Could Reach Half Protected  #C8EBB1   #00734C   

                COLOR_NNH    LICENSE  \
ecoregion month                        
13.0      5       #7BC141  CC-BY 4.0   
          6       #7BC141  CC-BY 4.0   
          7       #7BC141  CC-BY 4.0   
17.0      4       #7BC141  CC-BY 4.0   
          5       #7BC141  CC-BY 4.0   

                                                          geometry  \
ecoregion month                                                      
13.0      5      MULTIPOLYGON (((-13307108.288 7486619.094, -13...   
          6      MULTIPOLYGON (((-13307108.288 7486619.094, -13...   
          7      MULTIPOLYGON (((-13307108.288 7486619.094, -13...   
17.0      4      POLYGON ((-8394490.321 5288851.507, -8405347.7...   
          5      POLYGON ((-8394490.321 5288851.507, -8405347.7...   

                 occurrences  norm_occurrences  
ecoregion month                                 
13.0      5                2          0.000816  
          6                2          0.000918  
          7                2          0.001760  
17.0      4                2          0.000010  
          5             3023          0.001701

In [5]:
### activate panel and hvplot extensions you'll need for the interactive plot
pn.extension()
hv.extension("bokeh")

In [6]:
### plot the data

### get the plot bounds so they don't change with the slider
xmin, ymin, xmax, ymax = occurrence_gdf.total_bounds

### define the slider widget
slider = pn.widgets.DiscreteSlider(
    name = 'month', 
    options = {calendar.month_name[i]: i for i in range(1, 13)}
)

### plot occurrence by ecoregion and month
migration_plot = (
    occurrence_gdf
    .hvplot(
        ### color by number of veeries
        c = 'norm_occurrences',

        ### group the data by month so the slider can change the plot
        groupby = 'month',

        ### use background tiles
        geo = True, crs = ccrs.Mercator(), tiles = 'OSM',

        ### add a title
        title = "Veery Migration by Month",

        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),

        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,

        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### wrap  plot in the Panel layout
interactive_migration_plot = pn.panel(migration_plot)

### check it out
interactive_migration_plot

BokehModel(combine_events=True, render_bundle={'docs_json': {'2891f591-26d2-4d2b-9c85-ab489716096d': {'version…

Due to some annoying issues with permissions for the basemap, we can't export the map you just made as an .html file with the basemap. So we'll make another version, without a basemap, to export. We'll revisit this for your portfolio assignment to make a nice map with a basemap.

In [7]:
### make a path for your map to live
map_path = os.path.join(data_dir, "map_name.html")

### make a version for export, without the basemap
export_migration_plot = (
    occurrence_gdf
    .hvplot(
        ### color by number of veeries
        c = 'norm_occurrences',
        
        ### group the data by month so the slider can change the plot
        groupby = 'month',
        
        ### use background tiles
        #geo = True, crs = ccrs.Mercator(), tiles = 'OSM',
        
        ### add a title
        title = "Veery Migration by Month",
        
        ### set plot boundaries
        xlim = (xmin, xmax), ylim = (ymin, ymax),
        
        ### set dimensions of map
        frame_height = 600,
        frame_width = 900,
        
        ### put the widget at the bottom of the map
        widget_location = 'bottom',
        colorbar = False,
        widgets = {'column_name': slider}
    )
)

### export the map as an .html file
export_migration_plot.save(
    map_path,
    embed = True,
    resources = "inline",
)

### Reflect and Respond #2: Interpret your map
What seasonal trends do you see in the GBIF observations? How does your map align with the broader literature on Veery migration?

The main seasonal trend in the GBIF observations is a seasonal variation in migration, Verries are in South America during the heart of North American winter (Nov-Feb) and begin transitioning northward into Central America in March. April-October Veeries maintain a large presence in North America, with some presence in the Caribbean with the exception of the peak months of summer (June-July-Aug).